# பாடம் 13 - முகவர் நினைவு


## அமைப்பு

இந்த நோட்புக்கில் **Microsoft Agent Framework** (MAF) பயன்படுத்தி **நிறைவேற்றப்பட்ட நினைவகம்** கொண்ட பயணம் முன்பதிவு முகவர் எப்படி கட்டமைப்பது என்பதை காட்டுகிறது.

வேறுவிதமான முகவர் நினைவக வகைகள் — பணியாற்றும், குறுகிய கால, மற்றும் நீண்ட கால — எப்படி ஒரு முகவர் தகவலை உரையாடல்களில் தொடர்ந்து பிடித்து பயன்படுத்துகிறது என்பதைக் கற்றுக்கொள்ள நீங்கள் வாய்ப்பு பெறுவீர்கள்.

**முன்னிட்டு தேவையிகள்:**
- ஒரு Azure AI Foundry திட்டம், அதில் ஒரு செயல்படுத்தப்பட்ட உரையாடல் மாதிரி (உதா. `gpt-4o-mini`).
- Azure CLI உடன் உள்நுழைந்திருக்க வேண்டும் — உங்கள் டெர்மினலில் `az login` இயக்கவும்.
- `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Azure AI Foundry திட்டத்தின் எண்ட்பாயிண்ட்.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் செயல்படுத்தப்பட்ட மாதிரியின் பெயர்.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## முகவர் நினைவக வகைகள்

ஏஐ முகவர்கள் பல்வேறு வகையான நினைவகங்களை பயன்படுத்தக்கூடும், ஒவ்வொன்றும் தனித்துவமான நோக்கத்துடன் செயல்படுகிறது:

### வேலை நினைவகம்
அரட்டை தந்தி தான் — ஒரே அமர்வில் பரிமாறப்படும் செய்திகள். முகவர் ஒரே தந்தியில் உள்ள முந்தைய செய்திகளைக் குறிப்பிட்டு ஒருங்கிணைப்பை தக்க வைக்கலாம். MAF இல் இது **`agent.create_session()`** மூலம் உருவாக்கப்படுகின்றது, இது `AgentSession` ஐ திருப்பி வழங்கும்.

### குறுகிய கால நினைவகம்
ஒரு பணியை அல்லது அமர்வை முடியும் வரை நிலைக்கும் தகவல், ஆனால் நிரந்தரமாகச் சேமிக்கப்படாது. உதாரணமாக, முகவர் பல-தொகு திட்டமிடல் உரையாடலில் உண்மைகளை ஈட்டிக் கொண்டு இறுதி பயணம் திட்டத்தை உருவாக்கக்கூடும்.

### நீண்டகால நினைவகம்
**அமர்வுகளைத் தாண்டி** நிலைக்கும் முன்னுரிமைகள் மற்றும் உண்மைகள். திரும்ப வரும் பயனர் தங்களைப் பற்றிய உணவு கட்டுப்பாடுகள் அல்லது பயண முறைகளை மீண்டும் கூற வேண்டியதில்லை. நீண்டகால நினைவகம் பொதுவாக வெளிப்புற சேமிப்பகம் — தரவுத்தொகுப்பு, கோப்பு அல்லது வெக்டர் குறியீடு — மூலம் ஆதரிக்கப்படுகிறது மற்றும் கருவிகள் மூலம் முகவருக்கு வெளிப்படுகிறது.


## செயலாக்க நினைவகத்துடன் அமர்வுகள்

மிக எளிய நினைவக வடிவம் உரையாடல் அமர்வு ஆகும். நீங்கள் ஒரே அமர்வை (`agent.create_session()` மூலம் உருவாக்கப்பட்டது) தொடர்ச்சியாக `agent.run()` அழைப்புகளுக்கு அனுப்பும் பொழுது, முகவர் அந்த உரையாடலின் முழுமையான வரலாற்றை பார்க்க முடியும் மற்றும் முன்னைய விவரங்களை நினைவில் வைத்திருக்கும்.

ஒரு பயண முகவரியை உருவாக்கி செயலாக்க நினைவகத்தைக் காட்டுவோம்.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

காரணி முறைப்படி பட்ஜெட்டை நினைவுகொண்டு இருந்தது ஏனெனில் இரு செய்திகளும் ஒரே அமர்வை பகிர்கின்றன. இது **வெயரல் நினைவு** — இது அமர்வு முடிவுக்கு மட்டுமே தக்கிறது.

### புதிய திரெட்டுடன் என்ன நடக்கும்?

நாம் ஒரு **புதிய** அமர்வை உருவாக்கினால், காரணிக்கு முன்பு நடந்த உரையாடல் பற்றிய நினைவில்லை:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## நீண்ட கால நினைவுப்பொதி மாதிரியியல்

பயனர் விருப்பங்களை **அனைத்து அமர்வுகளிலும்** நினைவில் வைத்திருக்க, உரையாடல் நூலுக்கு வெளியே வாழும் நிலையான சேமிப்பகம் தேவையlidir. முகவர் இந்த சேமிப்பகத்தை அணுக **கருவிகளை** (tools) பயன்படுத்துகிறான் — தகவல்களை சேமிக்கவும் மீட்டெடுக்கவும் அழைக்கக்கூடிய செயல்பாடுகள்.

கீழே, எளிய நினைவக விருப்ப சேமிப்பகத்தை (உற்பத்தியில் நீங்கள் இதனை தரவுத்தளம் அல்லது வெக்டர் குறியீட்டுடன் ஆதரிக்கும்) செயல்படுத்தி, முகவர் பயன்படுத்தக்கூடிய கருவிகளாக வெளிப்படுத்துகிறோம்.

### கட்டமைப்பு
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — முதன்முறையாக பயனர் ஆண்டுவிழா பயணத்திற்கான முன்பதிவைக் செய்கிறார்

சாரா முதன்முறையாக வருகிறார். முகவர் கருவிகளின் மூலம் அவரது விருப்பங்களை சேமித்து அவற்றைப் பயன்படுத்தி ஹோட்டல்களை பரிந்துரைக்க வேண்டும்.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### சூழல் 2 — சில வாரங்களுக்கு பிறகு சாரா திரும்புகிறாள்

சாரா ஒரு **புதிய த்ரெட்டை** (புதிய அமர்வை மெய்ப்பிக்கும்) தொடங்குகிறாள். வேலை நினைவகம் காலியாக உள்ளது, ஆனால் நீண்ட கால முன்னுரிமை சேமிப்பகத்தில் இன்னும் 그녀 தகவல் உள்ளது. முகவர் அதை மீட்டெடுத்து பரிந்துரைகளை தனிப்பயனாக்க பயன்படுத்த வேண்டும்.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांशம்

இந்த பாடத்தில் நீங்கள் மூன்று வகையான முகவர் நினைவகங்கள் மற்றும் அவற்றை Microsoft Agent Framework மூலம் எவ்வாறு செயல்படுத்துவது என்பதை கற்றுக்கொண்டீர்கள்:

| நினைவக வகை | MAF முறை | ஆயுள் காலம் |
|---|---|---|
| **வேலை செய்யும்** | `agent.create_session()` | ஒரே உரையாடல் |
| **குறுகியகால** | ஒரு தாமரை உள்ளுள்ள தொடுக்குள் சேகரிக்கப்பட்ட சூழல் | ஒரே பணி / அமர்வு |
| **நீண்டகால** | வெளியினர் சேமிப்பகம் `@tool` செயல்களைப் பயன்படுத்தி அணுகபடும் | அமர்வுகளுக்கு மேல் |

### முக்கியக் கற்கைகள்
1. **`agent.create_session()`** வேலை செய்யும் நினைவகத்தை வழங்குகிறது — முகவர் அமர்வின் முழு உரையாடல் வரலாற்றையும் காண்கிறார்.
2. **புதிய அமர்வுகள் சூழலை இழக்கின்றன** — நீண்டகால நினைவகம் இல்லையென்றால் முகவர் கடந்த உரையாடல்களை நினைவில் வைக்க முடியாது.
3. **`@tool` செயலிகள்** இடையிலான இடைவெளியை நிரப்புகின்றன — அவைகள் முகவருக்குப் பொருளை நிலையான சேமிப்பகத்திலிருந்து சேமிக்க மற்றும் மீட்டு கொள்ள உதவுகின்றன.
4. **உருபத்தியாக்கல் நேரத்துடன் மேம்படும்** — அதிகமான விருப்பங்கள் சேமிக்கப்படுவதனால் முகவரின் பரிந்துரைகள் சிறந்ததாக மாறுகின்றன.

### பயன்பாடுகள் உண்மையில்
- **வாடிக்கையாளர் சேவை**: வாடிக்கையாளர் வரலாறு மற்றும் விருப்பங்களை நினைவில் வைக்க
- **தனிப்பட்ட உதவியாளர்கள்**: நாட்கள் அல்லது வாரங்கள் தாண்டி சூழலை பராமரிக்க
- **சுகாதாரம்**: நோயாளி தகவல் மற்றும் விருப்பங்களை கண்காணிக்க
- **மின் வணிகம்**: வரலாறின் அடிப்படையில் தனிப்பயன் கடையில் வாங்குதல்

### அடுத்த படிகள்
- நினைவக அகராதியை தரவுத்தளம் அல்லது வெக்டர் சேமிப்பகத்துடன் மாற்றுதல் (உதா. Azure AI Search)
- நேரம் சார்ந்த தகவல்களுக்கு நினைவக காலாவதியை சேர்த்தல்
- பகிரப்பட்ட நினைவகத்துடன் பல முகவர் அமைப்புகளை உருவாக்குதல்
- Cognee நோட்புக் மூலம் அறிவு-வரைபட ஆதரவு நினைவகத்தை ஆராய்ந்தல்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**தொகுப்புரை**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையான [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்க்காக முயற்சிப்போம் என்றாலும், தானாக இயங்கும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்க வாய்ப்பு உள்ளதைக் கவனிக்கவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரபூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பயன்படுத்த பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயன்பாட்டில் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்கு நாங்கள் பொறுப்பு கொள்ளமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
